In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)

In [2]:
df= pd.read_csv('/content/synthetic_noisy_dataset_40k_15col.csv')
print("Shape:", df.shape)
df.head()

Shape: (40200, 15)


,age,income,experience_years,education_years,hours_worked_per_week,satisfaction_score,department,city,performance_rating,projects_completed,training_hours,remote_work_ratio,tenure_years,overtime_hours,target_salary
0,58.0,69549.87,39.0,12.0,39.3,6.14,Engineering,Sylhet,2.0,5.0,35.9,0.497,39.0,2.8,68384.37
1,52.0,60693.22,32.0,16.0,29.5,7.23,Marketing,Dhaka,5.0,5.0,6.3,0.671,32.0,2.2,65594.91
2,43.0,45906.57,25.0,19.0,58.1,7.71,HR,Rajshahi,4.0,1.0,26.9,0.187,25.0,0.5,49246.79
3,58.0,65469.20,NaN,10.0,38.9,7.45,Engineering,Chittagong,4.0,3.0,13.3,0.255,39.0,13.7,72753.03
4,45.0,60345.96,24.0,17.0,34.6,4.05,Finance,Sylhet,4.0,8.0,7.1,0.612,22.0,0.0,NaN


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40200 entries, 0 to 40199
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   age                    38999 non-null  float64
 1   income                 38973 non-null  float64
 2   experience_years       38976 non-null  float64
 3   education_years        38993 non-null  float64
 4   hours_worked_per_week  39026 non-null  float64
 5   satisfaction_score     38976 non-null  float64
 6   department             40200 non-null  object 
 7   city                   40200 non-null  object 
 8   performance_rating     38999 non-null  float64
 9   projects_completed     38968 non-null  float64
 10  training_hours         38924 non-null  float64
 11  remote_work_ratio      38990 non-null  float64
 12  tenure_years           39009 non-null  float64
 13  overtime_hours         39023 non-null  float64
 14  target_salary          39014 non-null  float64
dtypes:

In [5]:
df.isnull().sum()

,0
age,1201
income,1227
experience_years,1224
education_years,1207
hours_worked_per_week,1174
satisfaction_score,1224
department,0
city,0
performance_rating,1201
projects_completed,1232


In [6]:
#Fill missing value with median value
num_col = df.select_dtypes(include=[np.number]).columns
df[num_col] = df[num_col].fillna(df[num_col].median())

#for alphabetic
alpha_col = df.select_dtypes(include=['object', 'category']).columns
for col in alpha_col:
    df[col] = df[col].fillna(df[col].mode()[0])
print(df.isnull().sum())
print(df.head())


age                      0
income                   0
experience_years         0
education_years          0
hours_worked_per_week    0
satisfaction_score       0
department               0
city                     0
performance_rating       0
projects_completed       0
training_hours           0
remote_work_ratio        0
tenure_years             0
overtime_hours           0
target_salary            0
dtype: int64
    age    income  experience_years  education_years  hours_worked_per_week  \
0  58.0  69549.87              39.0             12.0                   39.3   
1  52.0  60693.22              32.0             16.0                   29.5   
2  43.0  45906.57              25.0             19.0                   58.1   
3  58.0  65469.20              23.0             10.0                   38.9   
4  45.0  60345.96              24.0             17.0                   34.6   

   satisfaction_score   department        city  performance_rating  \
0                6.14  Engineering   

In [7]:
#outliners
num_col = df.select_dtypes(include=[np.number]).columns

for col in num_col:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

print(df.head())

    age    income  experience_years  education_years  hours_worked_per_week  \
0  58.0  69549.87              39.0             12.0                   39.3   
1  52.0  60693.22              32.0             16.0                   29.5   
2  43.0  45906.57              25.0             19.0                   58.1   
3  58.0  65469.20              23.0             10.0                   38.9   
4  45.0  60345.96              24.0             17.0                   34.6   

   satisfaction_score   department        city  performance_rating  \
0                6.14  Engineering      Sylhet                 2.0   
1                7.23    Marketing       Dhaka                 5.0   
2                7.71           HR    Rajshahi                 4.0   
3                7.45  Engineering  Chittagong                 4.0   
4                4.05      Finance      Sylhet                 4.0   

   projects_completed  training_hours  remote_work_ratio  tenure_years  \
0                 5.0         

In [8]:
#remove duplicate
print("Duplicate", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicate", df.duplicated().sum())


Duplicate 187
Duplicate 0


In [9]:
#invalid entires handle
#age
if 'age' in df.columns:
    df['age'] = np.where((df['age']<0)|(df['age']>100), np.nan, df['age'])

#income
if 'income' in df.columns:
    df['income'] = np.where(df['income']<0, np.nan, df['income'])

#experience_years
if 'experience_years' in df.columns:
    df['experience_years'] = np.where((df['experience_years']<0)|(df['experience_years']>60), np.nan, df['experience_years'])

#education_years
if 'education_years' in df.columns:
    df['education_years'] = np.where((df['education_years']<0)|(df['education_years']>30), np.nan, df['education_years'])

#performance_rating
if 'performance_rating' in df.columns:
    df['performance_rating'] = np.where((df['performance_rating']<1)|(df['performance_rating']>5), np.nan, df['performance_rating'])

#projects_completed
if 'projects_completed' in df.columns:
    df['projects_completed'] = np.where(df['projects_completed']<0, np.nan, df['projects_completed'])

#training_hours
if 'training_hours' in df.columns:
    df['training_hours'] = np.where(df['training_hours']<0, np.nan, df['training_hours'])

#remote_work_ratio
if 'remote_work_ratio' in df.columns:
    df['remote_work_ratio'] = np.where((df['remote_work_ratio']<0)|(df['remote_work_ratio']>1), np.nan, df['remote_work_ratio'])

#tenure_years
if 'tenure_years' in df.columns:
    df['tenure_years'] = np.where((df['tenure_years']<0)|(df['tenure_years']>60), np.nan, df['tenure_years'])

#overtime_hours
if 'overtime_hours' in df.columns:
    df['overtime_hours'] = np.where(df['overtime_hours']<0, np.nan, df['overtime_hours'])

#target_salary
if 'target_salary' in df.columns:
    df['target_salary'] = np.where(df['target_salary']<=0, np.nan, df['target_salary'])

print(df.isnull().sum())

age                      206
income                     0
experience_years           0
education_years            0
hours_worked_per_week      0
satisfaction_score         0
department                 0
city                       0
performance_rating         0
projects_completed         0
training_hours             0
remote_work_ratio          0
tenure_years               0
overtime_hours             0
target_salary              0
dtype: int64


##  Model Selection

`target_salary` is a continuous number, so this is a **regression** problem.

**Linear Regression** — the baseline. Fast, fully interpretable, and a fair test of whether the relationship between features and salary is roughly linear. If the fancier models don't beat this by much, it means the extra complexity isn't buying anything.

In [10]:
X = df.drop(columns=['target_salary'])
y = df['target_salary']

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, " Test:", X_test.shape)

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), numeric_cols),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols)
])

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
}

fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    print(f"{name} trained.")

Numeric columns: ['age', 'income', 'experience_years', 'education_years', 'hours_worked_per_week', 'satisfaction_score', 'performance_rating', 'projects_completed', 'training_hours', 'remote_work_ratio', 'tenure_years', 'overtime_hours']
Categorical columns: ['department', 'city']
Train: (32010, 14)  Test: (8003, 14)
Linear Regression trained.
Random Forest trained.
Gradient Boosting trained.


In [11]:
results = []

for name, pipe in fitted_pipelines.items():
    preds = pipe.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    results.append({'Model': name, 'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2})

results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
results_df

,Model,MAE,MSE,RMSE,R2
0,Gradient Boosting,4478.650872,3.548711e+07,5957.105452,0.870781
1,Random Forest,4514.001523,3.608079e+07,6006.728912,0.868620
2,Linear Regression,4604.526262,3.907492e+07,6250.993896,0.857717
